In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [19]:
df = pd.read_csv("data_FraudDetection_JAR2020.csv")
df

,fyear,gvkey,p_aaer,misstate,act,ap,at,ceq,che,cogs,...,soft_assets,ch_cs,ch_cm,ch_roa,issue,bm,dpi,reoa,EBIT,ch_fcf
0,1990,1009,NaN,0,10.047,3.736,32.335,6.262,0.002,30.633,...,0.312448,0.095082,0.082631,-0.019761,1,0.413170,0.873555,0.167620,0.161961,-0.042140
1,1990,1011,NaN,0,1.247,0.803,7.784,0.667,0.171,1.125,...,0.315904,0.188832,-0.211389,-0.117832,1,0.157887,0.745139,-0.428957,-0.157888,0.100228
2,1990,1017,NaN,0,55.040,3.601,118.120,44.393,3.132,107.343,...,0.605342,0.097551,-0.105780,0.091206,1,2.231337,1.015131,0.394768,0.063681,0.066348
3,1990,1021,NaN,0,24.684,3.948,34.591,7.751,0.411,31.214,...,0.793068,-0.005725,-0.249704,0.017545,1,1.043582,1.026261,0.094822,0.088347,-0.017358
4,1990,1028,NaN,0,17.325,3.520,27.542,-12.142,1.017,32.662,...,0.869182,-0.231536,-1.674893,-0.466667,0,-1.602508,0.598443,-0.942379,-0.700821,0.130349
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146040,2014,314866,NaN,0,262.600,12.400,1234.800,194.100,166.200,214.400,...,0.751944,0.560406,0.127217,-0.050591,1,0.103693,0.829680,-0.327178,-0.008179,-0.261606
146041,2014,315318,NaN,0,1578.400,106.700,4557.600,2459.600,997.300,324.400,...,0.742781,-0.118178,0.031360,0.095355,1,0.581796,0.743084,-0.077826,0.000461,-0.296702
146042,2014,316056,NaN,0,973.800,249.500,2015.900,-4.800,290.500,1185.500,...,0.751129,0.004207,-0.037925,0.072050,1,-0.000903,1.063878,-0.002877,0.153133,0.065569
146043,2014,317260,NaN,0,51.743,1.555,322.421,319.230,46.398,24.319,...,0.018001,NaN,NaN,NaN,1,1.109467,NaN,0.000000,0.028804,NaN


In [20]:
base_features = [

    "act", "ap", "at", "ceq", "che",
    "cogs", "csho", "dlc", "dltis",
    "dltt", "dp", "ib", "invt",
    "ivao", "ivst", "lct", "lt",
    "ni", "pstk", "re",
    "rect", "sale", "sstk", "txp",
    "txt", "xint", "prcc_f",
    "dch_wc", "ch_rsst",
    "dch_rec", "dch_inv",
    "soft_assets", "ch_cs",
    "ch_cm", "ch_roa",
    "issue", "bm", "dpi",
    "reoa", "EBIT", "ch_fcf"
]

In [21]:
df = df[["gvkey", "fyear", "misstate"] + base_features]
df = df.sort_values(["gvkey", "fyear"]).reset_index(drop=True)
df = df.replace([np.inf, -np.inf], np.nan)
print(f"{df['gvkey'].nunique():,} companies")
print(f"Fraud rate: {df['misstate'].mean():.4f}")

18,444 companies
Fraud rate: 0.0066


In [22]:
df["current_ratio"] = df["act"] / df["lct"]
df["debt_ratio"] = (df["dlc"] + df["dltt"]) / df["at"]
df["cash_ratio"] = df["che"] / df["lct"]
df["profit_margin"] = df["ni"] / df["sale"]
df["ebit_margin"] = df["EBIT"] / df["sale"]
df["asset_turnover"] = df["sale"] / df["at"]
df["retained_earnings_ratio"] = df["re"] / df["at"]
df["working_capital_ratio"] = (df["act"] - df["lct"]) / df["at"]

df = df.replace([np.inf, -np.inf], np.nan)

df

,gvkey,fyear,misstate,act,ap,at,ceq,che,cogs,csho,...,EBIT,ch_fcf,current_ratio,debt_ratio,cash_ratio,profit_margin,ebit_margin,asset_turnover,retained_earnings_ratio,working_capital_ratio
0,1004,1991,0,289.537,43.416,395.351,196.737,4.197,331.056,15.899,...,0.055586,-0.020650,3.137218,0.233534,0.045476,0.023707,1.315157e-04,1.069068,0.259703,0.498914
1,1004,1997,0,468.400,112.980,670.559,300.850,17.222,619.434,27.704,...,0.097905,-0.094099,3.140505,0.265071,0.115469,0.045590,1.251784e-04,1.166375,0.220100,0.476098
2,1004,2001,0,436.656,49.529,710.199,310.235,34.522,526.477,31.870,...,-0.110435,-0.040463,2.902063,0.366410,0.229437,-0.092277,-1.729006e-04,0.899355,0.196569,0.402974
3,1004,2002,0,396.412,51.485,686.621,294.988,29.154,496.747,31.851,...,0.000071,0.086275,1.947253,0.374171,0.143210,-0.020467,1.176969e-07,0.883074,0.180749,0.280849
4,1004,2003,0,432.204,57.582,709.292,301.684,41.010,523.302,32.245,...,0.028939,0.030194,3.292707,0.355366,0.312431,0.005375,4.438739e-05,0.919167,0.185938,0.424286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146040,315669,2013,0,352.205,43.810,616.867,390.715,179.651,0.000,146.496,...,-0.086688,NaN,4.250091,0.223521,2.167865,NaN,NaN,0.000000,-0.300467,0.436618
146041,316056,2013,0,923.200,211.300,1979.900,-86.800,267.600,1206.000,96.029,...,0.108793,0.563867,1.882161,0.678772,0.545566,0.014808,5.196722e-05,1.057377,-0.048588,0.218546
146042,316056,2014,0,973.800,249.500,2015.900,-4.800,290.500,1185.500,95.831,...,0.153133,0.065569,1.832863,0.627313,0.546772,0.082708,7.229033e-05,1.050796,-0.002877,0.219505
146043,317260,2014,0,51.743,1.555,322.421,319.230,46.398,24.319,23.431,...,0.028804,NaN,16.215293,0.000000,14.540270,0.131296,5.456431e-04,0.163727,0.000000,0.150586


In [23]:
df["distress_target"] = (
    (df["ni"] < 0) & (df["ch_fcf"] < 0) & (df["debt_ratio"] > 0.5)
).astype(int)

df["credit_risk_target"] = (
    (df["current_ratio"] < 1.0) & (df["debt_ratio"] > 0.4)
).astype(int)

df["earnings_manip_target"] = (
    (df["dch_rec"] > 0) &
    (df["soft_assets"] > df["soft_assets"].median()) &
    (df["ch_rsst"] > 0)
).astype(int)

print(f"Distress rate:       {df['distress_target'].mean():.4f}")
print(f"Credit risk rate:    {df['credit_risk_target'].mean():.4f}")
print(f"Earnings manip rate: {df['earnings_manip_target'].mean():.4f}")


Distress rate:       0.0294
Credit risk rate:    0.1061
Earnings manip rate: 0.2188


In [24]:
feature_cols = base_features + [
    "current_ratio", "debt_ratio", "cash_ratio", "profit_margin",
    "ebit_margin", "asset_turnover", "retained_earnings_ratio", "working_capital_ratio"
]

temporal_features = [
    "sales_growth", "inventory_growth", "receivable_growth",
    "asset_growth", "debt_growth", "margin_change", "roa_change"
]

feature_cols += temporal_features

target_cols = ["misstate", "distress_target", "credit_risk_target", "earnings_manip_target"]

df = df.dropna(subset=feature_cols[:-7] + target_cols)
print(f"After dropna: {len(df):,} rows")


After dropna: 122,915 rows


In [25]:
company_labels = df.groupby("gvkey")["misstate"].max()
fraud_companies = company_labels[company_labels == 1].index.values
clean_companies = company_labels[company_labels == 0].index.values

fraud_train, fraud_test = train_test_split(fraud_companies, test_size=0.2, random_state=42)
clean_train, clean_test = train_test_split(clean_companies, test_size=0.2, random_state=42)

train_ids = np.concatenate([fraud_train, clean_train])
test_ids = np.concatenate([fraud_test, clean_test])

train_df = df[df["gvkey"].isin(train_ids)].copy()
test_df = df[df["gvkey"].isin(test_ids)].copy()

print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")
print(f"Fraud companies - Train: {len(fraud_train)} | Test: {len(fraud_test)}")

Train: 98,051 rows | Test: 24,864 rows
Fraud companies - Train: 296 | Test: 75


In [26]:
scaler = RobustScaler()

train_df[feature_cols[:-7]] = train_df[feature_cols[:-7]].replace([np.inf, -np.inf], np.nan)
test_df[feature_cols[:-7]] = test_df[feature_cols[:-7]].replace([np.inf, -np.inf], np.nan)

train_df[feature_cols[:-7]] = scaler.fit_transform(train_df[feature_cols[:-7]])
test_df[feature_cols[:-7]] = scaler.transform(test_df[feature_cols[:-7]])

In [27]:
def split_contiguous(group):
    years = group["fyear"].values
    breaks = np.where(np.diff(years) != 1)[0]
    segments = []
    start = 0
    for b in breaks:
        segments.append(group.iloc[start:b+1].copy())
        start = b + 1
    segments.append(group.iloc[start:].copy())
    return segments

def add_temporal_features(segment):
    segment = segment.sort_values("fyear").copy()
    segment["sales_growth"] = segment["sale"].pct_change()
    segment["inventory_growth"] = segment["invt"].pct_change()
    segment["receivable_growth"] = segment["rect"].pct_change()
    segment["asset_growth"] = segment["at"].pct_change()
    segment["debt_growth"] = segment["dltt"].pct_change()
    segment["margin_change"] = segment["profit_margin"].diff()
    segment["roa_change"] = segment["ch_roa"].diff()
    return segment

In [28]:
def build_sequences(dataframe, seq_len=5):
    X, y_fraud, y_distress, y_credit, y_manip = [], [], [], [], []
    for _, company in dataframe.groupby("gvkey"):
        for segment in split_contiguous(company.sort_values("fyear")):
            segment = add_temporal_features(segment)
            segment = segment.replace([np.inf, -np.inf], np.nan)
            segment = segment.dropna(subset=feature_cols)
            if len(segment) < seq_len + 1:
                continue
            features = segment[feature_cols].values
            fraud = segment["misstate"].values
            distress = segment["distress_target"].values
            credit = segment["credit_risk_target"].values
            manip = segment["earnings_manip_target"].values
            for i in range(seq_len, len(segment)):
                X.append(features[i-seq_len:i])
                y_fraud.append(fraud[i])
                y_distress.append(distress[i])
                y_credit.append(credit[i])
                y_manip.append(manip[i])
    return (
        np.array(X, dtype=np.float32),
        np.array(y_fraud, dtype=np.float32),
        np.array(y_distress, dtype=np.float32),
        np.array(y_credit, dtype=np.float32),
        np.array(y_manip, dtype=np.float32)
    )

X_train, y_fraud_train, y_distress_train, y_credit_train, y_manip_train = build_sequences(train_df)
X_test, y_fraud_test, y_distress_test, y_credit_test, y_manip_test = build_sequences(test_df)

print(f"Train sequences: {len(X_train):,} | Test sequences: {len(X_test):,}")
print(f"Train fraud rate: {y_fraud_train.mean():.4f} | Test fraud rate: {y_fraud_test.mean():.4f}")

Train sequences: 34,227 | Test sequences: 8,679
Train fraud rate: 0.0079 | Test fraud rate: 0.0073


In [33]:
fraud_idx = np.where(y_fraud_train == 1)[0]
nonfraud_idx = np.where(y_fraud_train == 0)[0]
sampled_nonfraud = np.random.choice(nonfraud_idx, size=min(len(nonfraud_idx), len(fraud_idx) * 3), replace=False)
balanced_idx = np.concatenate([fraud_idx, sampled_nonfraud])
np.random.shuffle(balanced_idx)

'''X_train = X_train[balanced_idx]
y_fraud_train = y_fraud_train[balanced_idx]
y_distress_train = y_distress_train[balanced_idx]
y_credit_train = y_credit_train[balanced_idx]
y_manip_train = y_manip_train[balanced_idx]'''

print(f"Train size: {len(X_train):,}")
print(f"Fraud: {y_fraud_train.sum():.0f} | Non-fraud: {(y_fraud_train==0).sum():.0f}")


Train size: 34,227
Fraud: 271 | Non-fraud: 33956


In [30]:
class FraudDataset(Dataset):
    def __init__(self, X, y_fraud, y_distress, y_credit, y_manip):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_fraud = torch.tensor(y_fraud, dtype=torch.float32)
        self.y_distress = torch.tensor(y_distress, dtype=torch.float32)
        self.y_credit = torch.tensor(y_credit, dtype=torch.float32)
        self.y_manip = torch.tensor(y_manip, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y_fraud[idx], self.y_distress[idx], self.y_credit[idx], self.y_manip[idx]

train_loader = DataLoader(FraudDataset(X_train, y_fraud_train, y_distress_train, y_credit_train, y_manip_train), batch_size=256, shuffle=True)
test_loader = DataLoader(FraudDataset(X_test, y_fraud_test, y_distress_test, y_credit_test, y_manip_test), batch_size=256, shuffle=False)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


Train batches: 134 | Test batches: 34


In [31]:
from entmax import sparsemax

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.99, gamma=1.5, margin=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.margin = margin

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal = (self.alpha * ((1 - pt) ** self.gamma) * bce).mean()
        logits = logits.squeeze(1)
        targets = targets.squeeze(1)
        pos_logits = logits[targets == 1]
        neg_logits = logits[targets == 0]
        if len(pos_logits) > 0 and len(neg_logits) > 0:
            pairwise = self.margin - (pos_logits.unsqueeze(1) - neg_logits.unsqueeze(0))
            ranking_loss = torch.relu(pairwise).mean()
        else:
            ranking_loss = 0.0
        return focal + 0.4 * ranking_loss

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.GELU(), nn.LayerNorm(dim), nn.Dropout(0.1), nn.Linear(dim, dim)
        )
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(x + self.block(x))

class FraudModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.1),
            ResidualBlock(128),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.LayerNorm(64)
        )
        self.gru = nn.GRU(64, 128, num_layers=2, dropout=0.1, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(embed_dim=256, num_heads=8, batch_first=True)
        self.pool = nn.Sequential(nn.Linear(256, 64), nn.GELU(), nn.Linear(64, 1))
        self.shared = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.1)
        )
        self.fraud_head = nn.Linear(64, 1)
        self.distress_head = nn.Linear(64, 1)
        self.credit_head = nn.Linear(64, 1)
        self.manip_head = nn.Linear(64, 1)

    def forward(self, x):
        B, T, F = x.shape
        x = self.feature_extractor(x.reshape(B * T, F)).reshape(B, T, 64)
        x, _ = self.gru(x)
        attn_out, _ = self.attention(x, x, x)
        x = x + attn_out
        x = (x * sparsemax(self.pool(x), dim=1)).sum(dim=1)
        x = self.shared(x)
        return self.fraud_head(x), self.distress_head(x), self.credit_head(x), self.manip_head(x)

model = FraudModel(len(feature_cols)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-6)
fraud_loss_fn = FocalLoss()
aux_loss_fn = nn.BCEWithLogitsLoss()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Model parameters: 816,005


In [32]:
from sklearn.metrics import ndcg_score

best_ndcg = 0
patience = 50
patience_counter = 0

for epoch in range(500):
    model.train()
    total_loss = 0
    for X_batch, y_fraud, y_distress, y_credit, y_manip in train_loader:
        X_batch = X_batch.to(device)
        y_fraud = y_fraud.to(device).unsqueeze(1)
        y_distress = y_distress.to(device).unsqueeze(1)
        y_credit = y_credit.to(device).unsqueeze(1)
        y_manip = y_manip.to(device).unsqueeze(1)
        optimizer.zero_grad()
        fraud_pred, distress_pred, credit_pred, manip_pred = model(X_batch)
        loss = (fraud_loss_fn(fraud_pred, y_fraud)
                + 0.1 * aux_loss_fn(distress_pred, y_distress)
                + 0.1 * aux_loss_fn(credit_pred, y_credit)
                + 0.1 * aux_loss_fn(manip_pred, y_manip))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    fraud_probs, distress_probs, credit_probs, manip_probs = [], [], [], []
    with torch.no_grad():
        for X_batch, *_ in test_loader:
            fraud_pred, distress_pred, credit_pred, manip_pred = model(X_batch.to(device))
            fraud_probs.extend(torch.sigmoid(fraud_pred).cpu().numpy().flatten())
            distress_probs.extend(torch.sigmoid(distress_pred).cpu().numpy().flatten())
            credit_probs.extend(torch.sigmoid(credit_pred).cpu().numpy().flatten())
            manip_probs.extend(torch.sigmoid(manip_pred).cpu().numpy().flatten())

    fraud_probs = np.array(fraud_probs)
    distress_probs = np.array(distress_probs)
    credit_probs = np.array(credit_probs)
    manip_probs = np.array(manip_probs)

    k = int(0.01 * len(y_fraud_test))
    fraud_ndcg = ndcg_score(y_fraud_test.reshape(1, -1), fraud_probs.reshape(1, -1), k=k)
    fraud_roc = roc_auc_score(y_fraud_test, fraud_probs)
    fraud_pr = average_precision_score(y_fraud_test, fraud_probs)
    distress_roc = roc_auc_score(y_distress_test, distress_probs)
    credit_roc = roc_auc_score(y_credit_test, credit_probs)
    manip_roc = roc_auc_score(y_manip_test, manip_probs)

    print(f"Epoch {epoch+1:02d} | Loss {total_loss/len(train_loader):.4f} | "
          f"Fraud ROC {fraud_roc:.4f} | PR {fraud_pr:.4f} | NDCG@1% {fraud_ndcg:.4f} | "
          f"Distress ROC {distress_roc:.4f} | Credit ROC {credit_roc:.4f} | Manip ROC {manip_roc:.4f}")

    if fraud_ndcg > best_ndcg:
        best_ndcg = fraud_ndcg
        patience_counter = 0
        torch.save(model.state_dict(), "fraud.pt")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("\nEarly stopping triggered.")
        break


Epoch 01 | Loss 0.8276 | Fraud ROC 0.6832 | PR 0.0150 | NDCG@1% 0.0000 | Distress ROC 0.3535 | Credit ROC 0.3435 | Manip ROC 0.6092
Epoch 02 | Loss 0.5163 | Fraud ROC 0.7019 | PR 0.0148 | NDCG@1% 0.0139 | Distress ROC 0.4123 | Credit ROC 0.3464 | Manip ROC 0.5989
Epoch 03 | Loss 0.4179 | Fraud ROC 0.7514 | PR 0.0215 | NDCG@1% 0.0139 | Distress ROC 0.3945 | Credit ROC 0.3453 | Manip ROC 0.5960
Epoch 04 | Loss 0.4114 | Fraud ROC 0.7615 | PR 0.0216 | NDCG@1% 0.0209 | Distress ROC 0.4143 | Credit ROC 0.3657 | Manip ROC 0.5941
Epoch 05 | Loss 0.3807 | Fraud ROC 0.7231 | PR 0.0331 | NDCG@1% 0.0868 | Distress ROC 0.4347 | Credit ROC 0.3799 | Manip ROC 0.5804
Epoch 06 | Loss 0.3027 | Fraud ROC 0.7557 | PR 0.0203 | NDCG@1% 0.0130 | Distress ROC 0.4031 | Credit ROC 0.3595 | Manip ROC 0.5979
Epoch 07 | Loss 0.2735 | Fraud ROC 0.7217 | PR 0.0201 | NDCG@1% 0.0424 | Distress ROC 0.4839 | Credit ROC 0.4652 | Manip ROC 0.5959
Epoch 08 | Loss 0.2803 | Fraud ROC 0.7056 | PR 0.0187 | NDCG@1% 0.0331 | Dis

In [38]:
from sklearn.metrics import confusion_matrix, ndcg_score

model.load_state_dict(torch.load("fraud.pt", weights_only=True))
model.eval()

fraud_probs, distress_probs, credit_probs, manip_probs = [], [], [], []
with torch.no_grad():
    for X_batch, *_ in test_loader:
        fraud_pred, distress_pred, credit_pred, manip_pred = model(X_batch.to(device))
        fraud_probs.extend(torch.sigmoid(fraud_pred).cpu().numpy().flatten())
        distress_probs.extend(torch.sigmoid(distress_pred).cpu().numpy().flatten())
        credit_probs.extend(torch.sigmoid(credit_pred).cpu().numpy().flatten())
        manip_probs.extend(torch.sigmoid(manip_pred).cpu().numpy().flatten())

fraud_probs = np.array(fraud_probs)
distress_probs = np.array(distress_probs)
credit_probs = np.array(credit_probs)
manip_probs = np.array(manip_probs)

def get_confusion(y_true, probs, target_fpr):
    threshold = np.sort(probs)[::-1][int(target_fpr * len(probs))]
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return tp, fn, fp, tn, threshold

k = int(0.01 * len(y_fraud_test))
fraud_ndcg = ndcg_score(y_fraud_test.reshape(1, -1), fraud_probs.reshape(1, -1), k=k)
distress_ndcg = ndcg_score(y_distress_test.reshape(1, -1), distress_probs.reshape(1, -1), k=k)
credit_ndcg = ndcg_score(y_credit_test.reshape(1, -1), credit_probs.reshape(1, -1), k=k)
manip_ndcg = ndcg_score(y_manip_test.reshape(1, -1), manip_probs.reshape(1, -1), k=k)

ftp, ffn, ffp, ftn, fthr = get_confusion(y_fraud_test, fraud_probs, 0.1)
dtp, dfn, dfp, dtn, dthr = get_confusion(y_distress_test, distress_probs, 0.1)
ctp, cfn, cfp, ctn, cthr = get_confusion(y_credit_test, credit_probs, 0.1)
mtp, mfn, mfp, mtn, mthr = get_confusion(y_manip_test, manip_probs, 0.1)

print(f"Fraud    - ROC: {roc_auc_score(y_fraud_test, fraud_probs):.4f} | PR: {average_precision_score(y_fraud_test, fraud_probs):.4f} | NDCG@1%: {fraud_ndcg:.4f} | Caught: {ftp}/{ftp+ffn} ({100*ftp/(ftp+ffn):.1f}%) | FP: {ffp}/{ffp+ftn} ({100*ffp/(ffp+ftn):.1f}%)")
print(f"Distress - ROC: {roc_auc_score(y_distress_test, distress_probs):.4f} | PR: {average_precision_score(y_distress_test, distress_probs):.4f} | NDCG@1%: {distress_ndcg:.4f} | Caught: {dtp}/{dtp+dfn} ({100*dtp/(dtp+dfn):.1f}%) | FP: {dfp}/{dfp+dtn} ({100*dfp/(dfp+dtn):.1f}%)")
print(f"Credit   - ROC: {roc_auc_score(y_credit_test, credit_probs):.4f} | PR: {average_precision_score(y_credit_test, credit_probs):.4f} | NDCG@1%: {credit_ndcg:.4f} | Caught: {ctp}/{ctp+cfn} ({100*ctp/(ctp+cfn):.1f}%) | FP: {cfp}/{cfp+ctn} ({100*cfp/(cfp+ctn):.1f}%)")
print(f"Manip    - ROC: {roc_auc_score(y_manip_test, manip_probs):.4f} | PR: {average_precision_score(y_manip_test, manip_probs):.4f} | NDCG@1%: {manip_ndcg:.4f} | Caught: {mtp}/{mtp+mfn} ({100*mtp/(mtp+mfn):.1f}%) | FP: {mfp}/{mfp+mtn} ({100*mfp/(mfp+mtn):.1f}%)")
print(f"Actual Fraud Rate: {y_fraud_test.mean():.4f}")


Fraud    - ROC: 0.6930 | PR: 0.0326 | NDCG@1%: 0.0912 | Caught: 20/63 (31.7%) | FP: 848/8616 (9.8%)
Distress - ROC: 0.7942 | PR: 0.0930 | NDCG@1%: 0.1397 | Caught: 90/226 (39.8%) | FP: 778/8453 (9.2%)
Credit   - ROC: 0.8848 | PR: 0.4245 | NDCG@1%: 0.5963 | Caught: 376/773 (48.6%) | FP: 492/7906 (6.2%)
Manip    - ROC: 0.7057 | PR: 0.4137 | NDCG@1%: 0.6678 | Caught: 414/2079 (19.9%) | FP: 454/6600 (6.9%)
Actual Fraud Rate: 0.0073
